# Selenium 웹 크롤링 기초

## 1. Selenium 소개

### Selenium이란?
Selenium은 웹 브라우저를 자동화하는 도구입니다. 웹 테스트 자동화, 데이터 수집, 웹 스크래핑 등 다양한 용도로 사용됩니다.

### 왜 Selenium을 사용하나요?
* 동적 웹페이지 처리 가능
* 실제 브라우저 동작 시뮬레이션
* JavaScript로 생성되는 콘텐츠 수집 가능

### 공식문서
- Selenium 공식 문서: https://www.selenium.dev/documentation/
- Python Selenium 튜토리얼: https://selenium-python.readthedocs.io/

 ## 1. 환경 설정

 필요한 패키지를 설치하고 import 합니다.

In [1]:
# pip install selenium webdriver-manager


 ### 필요한 라이브러리 import

In [2]:
from selenium import webdriver  # 웹드라이버 기본 모듈
from selenium.webdriver.chrome.service import Service  # 크롬 서비스 관리 모듈
from selenium.webdriver.chrome.options import Options  # 크롬 브라우저 옵션 설정 모듈
from webdriver_manager.chrome import ChromeDriverManager  # 크롬드라이버 자동 관리 모듈
from selenium.webdriver.common.by import By  # 요소 찾기 방법 지정 모듈
from selenium.webdriver.support.ui import WebDriverWait  # 요소 대기 관리 모듈
from selenium.webdriver.support import expected_conditions as EC  # 예상 조건 모듈
from selenium.webdriver.common.keys import Keys  # 키보드 입력 모듈
from selenium.common.exceptions import TimeoutException, NoSuchElementException  # 예외 처리 모듈
import time  # 시간 대기 모듈
import pandas as pd  # 데이터 처리 모듈

 ## 2. 기본 브라우저 실행



 ### 방법 1: Selenium 4.6+ 자동 드라이버 관리 (가장 간단)

In [3]:
# 최신 방법 1: Selenium 내장 자동 관리 (권장 - 간단한 프로젝트)
chrome_options = Options()
chrome_options.add_argument('--start-maximized')  # 브라우저를 최대화하여 시작

# Selenium이 자동으로 ChromeDriver를 다운로드하고 관리
driver = webdriver.Chrome(options=chrome_options)

# 웹페이지 접속
driver.get("https://www.google.com")

# 현재 페이지의 제목 출력
print("페이지 제목:", driver.title)

페이지 제목: Google


In [4]:
# 브라우저 종료
driver.quit() 

 ## 3. 고급 Chrome 옵션 설정 (봇 감지 회피)

In [5]:
def get_chrome_driver():
    """봇 감지를 회피하는 최적화된 Chrome 드라이버 생성"""
    
    chrome_options = Options()
    
    # ========== 기본 옵션 ==========
    # 브라우저를 최대화 상태로 시작 (전체 화면)
    chrome_options.add_argument('--start-maximized')
    
    # Selenium이 브라우저를 제어하고 있다는 표시를 숨김
    # (웹사이트가 자동화 도구를 감지하지 못하도록 함)
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    
    # ========== 자동화 감지 회피 ==========
    # 개발자 도구에서 "Chrome이 자동화된 테스트 소프트웨어에 의해 제어되고 있습니다" 메시지 제거
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    
    # 자동화 확장 프로그램 사용 안 함
    chrome_options.add_experimental_option('useAutomationExtension', False)
    
    # ========== User-Agent 설정 ==========
    # 브라우저의 신원 정보를 일반 사용자의 Chrome 브라우저로 위장
    # (봇이 아닌 실제 사용자처럼 보이게 함)
    user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    chrome_options.add_argument(f'user-agent={user_agent}')

    # ========== 드라이버 생성 ==========
    driver = webdriver.Chrome(options=chrome_options)
    
    # ========== JavaScript를 통한 추가 위장 ==========
    # Chrome DevTools Protocol을 사용하여 navigator.webdriver 속성을 제거
    # 웹사이트에서 JavaScript로 'navigator.webdriver'를 확인할 때 undefined를 반환
    # (많은 웹사이트가 이 속성으로 Selenium을 감지함)
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': '''
            Object.defineProperty(navigator, 'webdriver', {
                get: () => undefined
            })
        '''
    })
    
    return driver


driver = get_chrome_driver()
driver.get("https://www.google.com")
print("드라이버 준비 완료!")
time.sleep(2)
driver.quit()

드라이버 준비 완료!


 ## 4. 웹 요소 찾기 (최신 방법 - By 클래스 사용)



 **중요**: Selenium 4.x에서는 반드시 `By` 클래스를 사용해야 합니다!

In [6]:
driver = get_chrome_driver()
driver.get("https://www.google.com")

# 올바른 방법: By 클래스 사용
# 1. ID로 요소 찾기
# element = driver.find_element(By.ID, "element_id")

# 2. NAME으로 요소 찾기
search_box = driver.find_element(By.NAME, "q")

# 3. CSS 선택자로 요소 찾기
# element = driver.find_element(By.CSS_SELECTOR, ".class-name")

# 4. CLASS_NAME으로 요소 찾기
# element = driver.find_element(By.CLASS_NAME, "class-name")

# 5. XPath로 요소 찾기
# element = driver.find_element(By.XPATH, "//div[@id='search']")

# 6. TAG_NAME으로 요소 찾기
# elements = driver.find_elements(By.TAG_NAME, "a")

# 7. LINK_TEXT로 요소 찾기
# element = driver.find_element(By.LINK_TEXT, "로그인")

# 8. PARTIAL_LINK_TEXT로 요소 찾기
# element = driver.find_element(By.PARTIAL_LINK_TEXT, "로그")

# 검색어 입력 및 실행
search_box.clear()  # 기존 텍스트 제거
search_box.send_keys("Python Selenium")
search_box.send_keys(Keys.RETURN)  # Enter 키 입력

time.sleep(5)
print("검색 완료:", driver.title)
driver.quit()

검색 완료: Python Selenium - Google 검색


### XPath란?
- 웹페이지에서 원하는 요소를 찾기 위한 **주소 표기법**입니다.

**언제 사용하나요?**
- ID나 Class가 없을 때
- 복잡한 위치의 요소를 찾을 때
- 텍스트 내용으로 찾아야 할 때

간단한 경우는 ID나 CSS Selector가 더 편하고, 복잡한 경우에 XPath를 사용합니다!

 ## 5. 대기(Wait) 처리 방법



 ### 5.1 명시적 대기 (Explicit Wait) - 권장

 특정 조건이 만족될 때까지 대기

In [7]:
driver = get_chrome_driver()
driver.get("https://www.google.com")

# WebDriverWait 객체 생성 (최대 10초 대기)
wait = WebDriverWait(driver, 10)

# 1. presence_of_element_located: 요소가 DOM에 존재할 때까지 대기
# - 요소가 화면에 보이지 않아도 됨
# 안전장치: (거의) 무조건 써야함
search_box = wait.until(
    EC.presence_of_element_located((By.NAME, "q"))
)
search_box.send_keys("데이터 분석")

# 2. element_to_be_clickable: 요소가 클릭 가능할 때까지 대기
# - 요소가 보이고 활성화 상태여야 함
search_button = wait.until(
    EC.element_to_be_clickable((By.NAME, "q"))
)
search_button.click()

# 3. visibility_of_element_located: 요소가 화면에 보일 때까지 대기
# - 요소가 DOM에 존재하고 화면에 표시되어야 함
results = wait.until(
    EC.visibility_of_element_located((By.ID, "rso"))
)

print("검색 결과 로드 완료!")
time.sleep(2)
driver.quit()

TimeoutException: Message: 
Stacktrace:
0   chromedriver                        0x0000000102843cf8 cxxbridge1$str$ptr + 2895872
1   chromedriver                        0x000000010283bc34 cxxbridge1$str$ptr + 2862908
2   chromedriver                        0x0000000102361570 _RNvCs47EqcsrPRmA_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 74324
3   chromedriver                        0x00000001023a8f34 _RNvCs47EqcsrPRmA_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 367640
4   chromedriver                        0x00000001023ea3d8 _RNvCs47EqcsrPRmA_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 635068
5   chromedriver                        0x000000010239d0f8 _RNvCs47EqcsrPRmA_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 318940
6   chromedriver                        0x000000010280781c cxxbridge1$str$ptr + 2648868
7   chromedriver                        0x000000010280adf8 cxxbridge1$str$ptr + 2662656
8   chromedriver                        0x00000001027e8334 cxxbridge1$str

 ## 6. 스크롤 처리 방법

In [ ]:
driver = get_chrome_driver()
driver.get("https://play.google.com/store/apps?hl=ko")

# 1. 특정 픽셀만큼 스크롤
driver.execute_script("window.scrollTo(0, 500);")
print("500픽셀 스크롤 완료")

500픽셀 스크롤 완료


In [ ]:
# 2. 부드러운 스크롤 (점진적 스크롤)
for i in range(5):
    driver.execute_script(f"window.scrollTo(0, {500 + i * 100});")
    time.sleep(0.5)
print("부드러운 스크롤 완료")

부드러운 스크롤 완료


In [ ]:
# 3. 페이지 맨 아래로 스크롤
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
print("페이지 끝까지 스크롤 완료")

페이지 끝까지 스크롤 완료


In [ ]:
# 4. 페이지 맨 위로 스크롤
driver.execute_script("window.scrollTo(0, 0);")
print("페이지 처음으로 이동")

페이지 처음으로 이동


In [ ]:
driver.quit()

 ## 7. JavaScript 실행 예시

In [ ]:
driver = get_chrome_driver()
driver.get("https://play.google.com/store/apps?hl=ko")

wait = WebDriverWait(driver, 10)

In [ ]:
# 1. JavaScript로 요소 클릭 (일반 click()이 작동하지 않을 때 유용)
try:
    # TV 탭 클릭

    element = wait.until(
        EC.presence_of_element_located((By.XPATH, '//*[@id="yDmH0d"]/c-wiz[2]/div/div/c-wiz/div/div/div/div[1]/div[3]/a'))
    )
    
    # 클릭 방법 1
    # element.click() # 클릭 함수 종종 동작 안 됨
    
    # 클릭 방법 2
    driver.execute_script("arguments[0].click();", element)
    print("JavaScript 클릭 성공")
except Exception as e:
    print(f"요소 찾기 실패: {e}")

time.sleep(2)


JavaScript 클릭 성공


In [ ]:
# 2. JavaScript로 Alert 띄우기
driver.execute_script("alert('Hello from Selenium!');")
time.sleep(2)

# Alert 처리
alert = driver.switch_to.alert
alert.accept()  # 확인 버튼 클릭

In [ ]:
# 3. JavaScript로 스크롤
driver.execute_script("window.scrollTo(0, 1000);")
print("JavaScript 스크롤 완료")

time.sleep(1)
driver.quit()

1. url로 해당 페이지 접속
2. 내가 원하는 요소를 찾고
3. 해당 요소에 키보드 입력 혹은 클릭
4. 혹은 내가 원하는 요소를 찾아서 원하는 정보를 수집

---

 ## 실전 예제 : 네이버 검색 결과 수집

![name:query](../../../images/screenshot%202025-10-29%20오후%208.19.14.png)

In [ ]:
driver = get_chrome_driver()
wait = WebDriverWait(driver, 10)
from pprint import pprint


# 네이버 접속
driver.get("https://www.naver.com")

# 검색창 찾기 및 검색어 입력
search_box = wait.until(
    EC.presence_of_element_located((By.NAME, "query"))
)

In [ ]:
search_box.send_keys("인공지능")
search_box.send_keys(Keys.RETURN)

![](../../../images/screenshot%202025-10-29%20오후%208.24.56.png)

In [ ]:
# 뉴스 탭 클릭
news_button = wait.until(
    EC.element_to_be_clickable((By.XPATH, '//*[@id="lnb"]/div[1]/div/div[1]/div/div[1]/div[2]/a'))
)
news_button.click()

# 뉴스 검색 결과 대기 
search_results = wait.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".KetQP_LKIbcM61v6xsK_")) 
)

In [ ]:
# 모든 뉴스 기사 정보를 저장할 리스트
news_articles = []

print(f"\n{'='*80}")
print(f"총 {len(search_results)}개의 뉴스 기사 발견")
print(f"{'='*80}\n")

# 각 뉴스 기사 정보 추출
for idx, article in enumerate(search_results, 1):
    article_info = {
        'press': None,
        'title': None,
        'content': None,
        'url': None,
    }
    
    try:
        # 언론사명
        press = article.find_element(By.CSS_SELECTOR, ".sds-comps-profile-info-title-text")
        article_info['press'] = press.text
    except:
        pass
    
    try:
        # 제목
        title = article.find_element(By.CSS_SELECTOR, ".sds-comps-text-type-headline1")
        article_info['title'] = title.text
    except:
        pass
    
    try:
        # 본문 미리보기
        content = article.find_element(By.CSS_SELECTOR, ".sds-comps-text-ellipsis-3")
        article_info['content'] = content.text
    except:
        pass
    
    try:
        # 기사 URL
        url_element = article.find_element(By.CSS_SELECTOR, ".oE0MWYkMadhMOexVagqP")
        article_info['url'] = url_element.get_attribute('href')
    except:
        pass
    
    
    # 리스트에 저장
    news_articles.append(article_info)
    
    # 개별 기사 출력
    pprint(article_info)
    print()


총 10개의 뉴스 기사 발견

{'content': '윤정민 이현주 기자 = 최수연 네이버 대표가 글로벌 기업 리더들과 각국 정부 주요 관계자가 모인 자리에서 네이버의 '
            '풀스택 인공지능(AI) 구축·운영 경험과 혁신 방향을 소개한 가운데 AI 데이터센터에 대한 지원책(세제 혜택, '
            '행정절차 간소화 등) 필요성을 강조했다. 최 대표는 29일 오전 경북 경주...',
 'press': '뉴시스',
 'title': '최수연 네이버 대표 "AI데이터센터 세제 혜택·규제 완화 필요"[경주 APE...',
 'url': 'https://www.newsis.com/view/NISX20251029_0003381441'}

{'content': '이재명 대통령이 "오늘날 지속 가능한 발전을 이끌 혁신의 핵심은 바로 인공지능"이라며 "APEC 정상회의에서 AI '
            '이니셔티브를 제안할 것"이라고 밝혔습니다. 이 대통령은 오늘(29일) \'APEC(아시아태평양경제협력체) CEO '
            '서밋\' 기조 연설에서 "\'모두의 AI 비전이 APEC 뉴노멀로 자리잡길 기대한다"며 이같이...',
 'press': '채널A',
 'title': '이 대통령 “APEC 정상회의서 인공지능 이니셔티브 제안할 것”',
 'url': 'https://www.ichannela.com/news/main/news_detailPage.do?publishId=000000498247'}

{'content': '이투데이=김연진 기자 | LG유플러스가 경희대학교와 협력해 네트워크 트래픽 데이터가 발생한 지역의 특성을 판별하는 '
            '인공지능(AI) 모델을 개발했다고 29일 밝혔다. LG유플러스는 별도 현장 조사나 긴 테스트 없이 필요한 정보를 '
            '빠르게 제공하는 AI 모델을 개발하면서 네트워크 기술 경쟁력을 확보하게 됐다....',
 'press': '이투데이',
 'tit

In [ ]:
driver.quit()

### 무한 스크롤 페이지의 데이터 수집

In [ ]:
driver = get_chrome_driver()
wait = WebDriverWait(driver, 10)
from pprint import pprint

try:
    # 네이버 접속
    driver.get("https://www.naver.com")

    # 검색창 찾기 및 검색어 입력
    search_box = wait.until(
        EC.presence_of_element_located((By.NAME, "query"))
    )
    search_box.send_keys("인공지능")
    search_box.send_keys(Keys.RETURN)

    # 뉴스 탭 클릭
    news_button = wait.until(
        EC.element_to_be_clickable((By.XPATH, '//*[@id="lnb"]/div[1]/div/div[1]/div/div[1]/div[2]/a'))
    )
    news_button.click()

    # 뉴스 검색 결과 대기 
    wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".KetQP_LKIbcM61v6xsK_")) 
    )

    # 모든 뉴스 기사 정보를 저장할 리스트
    news_articles = []
    
    # 중복 방지를 위한 URL 세트
    seen_urls = set()
    
    # 스크롤 설정
    scroll_pause_time = 2  # 스크롤 후 대기 시간
    max_scrolls = 10  # 최대 스크롤 횟수 (원하는 만큼 조정)
    scroll_count = 0
    
    print(f"\n{'='*80}")
    print(f"뉴스 수집 시작 (최대 {max_scrolls}회 스크롤)")
    print(f"{'='*80}\n")

    while scroll_count < max_scrolls:
        # 현재 페이지의 모든 뉴스 기사 찾기
        search_results = driver.find_elements(By.CSS_SELECTOR, ".KetQP_LKIbcM61v6xsK_")
        
        # 각 뉴스 기사 정보 추출
        for article in search_results:
            article_info = {
                'press': None,
                'title': None,
                'content': None,
                'url': None,
            }
            
            try:
                # 기사 URL 먼저 확인 (중복 체크용)
                url_element = article.find_element(By.CSS_SELECTOR, ".oE0MWYkMadhMOexVagqP")
                article_url = url_element.get_attribute('href')
                
                # 이미 수집한 기사면 건너뛰기
                if article_url in seen_urls:
                    continue
                
                article_info['url'] = article_url
                seen_urls.add(article_url)
                
            except:
                continue
            
            try:
                # 언론사명
                press = article.find_element(By.CSS_SELECTOR, ".sds-comps-profile-info-title-text")
                article_info['press'] = press.text
            except:
                pass
            
            try:
                # 제목
                title = article.find_element(By.CSS_SELECTOR, ".sds-comps-text-type-headline1")
                article_info['title'] = title.text
            except:
                pass
            
            try:
                # 본문 미리보기
                content = article.find_element(By.CSS_SELECTOR, ".sds-comps-text-ellipsis-3")
                article_info['content'] = content.text
            except:
                pass
            
            # 리스트에 저장
            news_articles.append(article_info)
            
            # 개별 기사 출력
            print(f"[{len(news_articles)}] 새 기사 수집")
            pprint(article_info)
            print()
        
        # 현재까지 수집한 기사 수 출력
        print(f"{'='*80}")
        print(f"현재까지 수집된 기사: {len(news_articles)}개")
        print(f"{'='*80}\n")
        
        # 페이지 끝으로 스크롤
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        
        # 페이지 로딩 대기
        time.sleep(scroll_pause_time)

        scroll_count += 1
        print(f"스크롤 {scroll_count}/{max_scrolls} 완료\n")
    
    # 최종 결과 출력
    print(f"\n{'='*80}")
    print(f"수집 완료! 총 {len(news_articles)}개의 뉴스 기사 수집")
    print(f"{'='*80}\n")
    
except TimeoutException:
    print("검색 결과 로딩 시간 초과")
except Exception as e:
    print(f"오류 발생: {e}")
    import traceback
    traceback.print_exc()
finally:
    time.sleep(3)
    driver.quit()


뉴스 수집 시작 (최대 10회 스크롤)

[1] 새 기사 수집
{'content': '앞으로는 전 금융권에 걸친 범죄 계좌를 인공지능(AI) 플랫폼을 활용해 신속하게 정지할 수 있게 된다. 금융위원회는 '
            '29일 오후 경기도 용인 금융보안원에서 ‘인공지능 정보공유·분석 플랫폼 에이샙(ASAP)’ 출범식을 열고, '
            '은행·저축은행 등 금융사 130곳이 보이스피싱 관련 90여개 정보를 실시간으로 공유한다고...',
 'press': '한겨레',
 'title': '인공지능이 보이스피싱 ‘땅굴 계좌’ 즉시 차단한다',
 'url': 'https://www.hani.co.kr/arti/economy/finance/1226120.html'}

[2] 새 기사 수집
{'content': '윤정민 이현주 기자 = 최수연 네이버 대표가 글로벌 기업 리더들과 각국 정부 주요 관계자가 모인 자리에서 네이버의 '
            '풀스택 인공지능(AI) 구축·운영 경험과 혁신 방향을 소개한 가운데 AI 데이터센터에 대한 지원책(세제 혜택, '
            '행정절차 간소화 등) 필요성을 강조했다. 최 대표는 29일 오전 경북 경주...',
 'press': '뉴시스',
 'title': '최수연 네이버 대표 "AI데이터센터 세제 혜택·규제 완화 필요"[경주 APE...',
 'url': 'https://www.newsis.com/view/NISX20251029_0003381441'}

[3] 새 기사 수집
{'content': "전북 전주시는 29일 전주정보문화산업진흥원에서 인공지능(AI) 추진위원회 출범식과 'AI 대전환, 전주 AX미래전략 "
            "포럼'을 동시 개최했다. 인공지능 추진위는 인공지능과 로봇, 모빌리티, 바이오, 보안 분야의 산학연 전문가 20명으로 "
            '구성됐으며, 앞으로 전주시 AI 정책의 지휘부 역할을 수행한다. 위원장은...',
 'press': '연합뉴

### 데이터 저장

In [ ]:
from datetime import datetime

df = pd.DataFrame(news_articles)
    
# 데이터프레임 정보 출력
print("[데이터 미리보기]")
display(df.head())

# ========== 파일 저장 ==========
# 현재 시간을 파일명에 포함
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# JSON 저장
json_filename = f"naver_news_{timestamp}.json"
df.to_json(json_filename, orient='records', force_ascii=False, indent=2)
print(f"\nJSON 저장 완료: {json_filename}")